## Experimenting DkNN

In [1]:
# python stuff

from pathlib import Path as Path
from numpy.random import randint

# Our stuff
from datasets.cifar import Cifar
from models.model_wrap import ModelWrap

# from credibility import get_credibility

# torch stuff
import torch
from torchvision.models import vgg16, VGG16_Weights
from peepholes.peepholes import Peepholes
from peepholes.svd_peepholes import peep_matrices_from_svds as parser_fn
from credibility.DkNN import NearestNeighbor, DkNN


use_cuda = torch.cuda.is_available()
cuda_index = torch.cuda.device_count() - 2
device = torch.device(f"cuda:{cuda_index}" if use_cuda else "cpu")
print(f"Using {device} device")

#--------------------------------
# Dataset 
#--------------------------------
# model parameters
dataset = 'CIFAR100' 
seed = 29
bs = 64
data_path = '/srv/newpenny/XAI/LM/data/CIFAR100'

ds = Cifar(dataset=dataset, data_path=data_path)
ds.load_data(
        batch_size = bs,
        data_kwargs = {'num_workers': 4, 'pin_memory': True},
        seed = seed,
        )

Using cuda:6 device
dataset: CIFAR100
Files already downloaded and verified
Files already downloaded and verified


{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad0ac830>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4aad051c40>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aad0ad4c0>}

In [2]:
#--------------------------------
# Model 
#--------------------------------
pretrained = True
model_dir = '/srv/newpenny/XAI/LM/models'
model_name = f'vgg16_pretrained={pretrained}_dataset={dataset}-'\
f'augmented_policy=CIFAR10_bs={bs}_seed={seed}.pth'

nn = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
in_features = 4096
num_classes = len(ds.get_classes()) 
nn.classifier[-1] = torch.nn.Linear(in_features, num_classes)
model = ModelWrap(device=device)
model.set_model(model=nn, path=model_dir, name=model_name, verbose=True)

layers_dict = {'classifier': [0]}
              
model.set_target_layers(target_layers=layers_dict, verbose=True)
print('target layers: ', model.get_target_layers()) 

direction = {'save_input':True, 'save_output':False}
model.add_hooks(**direction, verbose=False) 

dry_img, _ = ds._train_ds.dataset[0]
dry_img = dry_img.reshape((1,)+dry_img.shape)
model.dry_run(x=dry_img)

#--------------------------------
# SVDs 
#--------------------------------
svds_path = Path.cwd()/'../data/svdsBanana'
svds_name = 'svdsBatata' 
model.get_svds(model=model, path=svds_path, name=svds_name, verbose=True)
for k in model._svds.keys():
    print('svd shapes: ', k, model._svds[k]['Vh'].shape)
#--------------------------------
# Peepholes 
#--------------------------------
phs_name = 'peepholes'
phs_dir = Path.cwd()/'../data/peepholes'
peepholes = Peepholes(
        path = phs_dir,
        name = phs_name,
        )
loaders = ds.get_dataset_loaders()
# copy dataset to peepholes dataset
peepholes.get_peep_dataset(
        loaders = loaders,
        verbose = True
        ) 

peepholes.get_activations(
        model=model,
        loaders=loaders,
        verbose=True
        )

peepholes.get_peepholes(
        model = model,
        peep_matrices = model._svds,
        parser = parser_fn,
        verbose = True
        )

/home/lorenzocapelli/repos/XAI/src/models/model_wrap.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self._checkpoint = torch.load(file, map_location=self.device)



-----------------
checkpoint
-----------------
state_dict keys: 
 odict_keys(['features.0.weight', 'features.0.bias', 'features.2.weight', 'features.2.bias', 'features.5.weight', 'features.5.bias', 'features.7.weight', 'features.7.bias', 'features.10.weight', 'features.10.bias', 'features.12.weight', 'features.12.bias', 'features.14.weight', 'features.14.bias', 'features.17.weight', 'features.17.bias', 'features.19.weight', 'features.19.bias', 'features.21.weight', 'features.21.bias', 'features.24.weight', 'features.24.bias', 'features.26.weight', 'features.26.bias', 'features.28.weight', 'features.28.bias', 'classifier.0.weight', 'classifier.0.bias', 'classifier.3.weight', 'classifier.3.bias', 'classifier.6.weight', 'classifier.6.bias']) 

train_loss 1.4151224618911744
val_loss 0.8791644280883157
train_accuracy 62.295
val_accuracy 74.96000000000001
epoch 59
batch_size 64
lr 0.001
-----------------

target layers:  {'classifier.0': Linear(in_features=25088, out_features=4096, bias=Tru

In [3]:
ds.config

{'num_classes': 100,
 'input_ch': 3,
 'means': (0.438, 0.418, 0.377),
 'stds': (0.3, 0.287, 0.294)}

In [4]:
peepholes._peepds

{'train': TensorDict(
     fields={
         image: MemoryMappedTensor(shape=torch.Size([40000, 3, 224, 224]), device=cpu, dtype=torch.float32, is_shared=True),
         in_activations: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 25088]), device=cpu, dtype=torch.float32, is_shared=True),
                 classifier.3: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch.float32, is_shared=True),
                 features.28: MemoryMappedTensor(shape=torch.Size([40000, 512, 14, 14]), device=cpu, dtype=torch.float32, is_shared=True)},
             batch_size=torch.Size([40000]),
             device=cpu,
             is_shared=False),
         label: MemoryMappedTensor(shape=torch.Size([40000]), device=cpu, dtype=torch.float32, is_shared=True),
         peepholes: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch

In [5]:
batch_dict = {key : value for key, value in peepholes._n_samples.items()}
kwargs = {'batch_dict': batch_dict,
          'verbose': True}
ph_dl = peepholes.get_dataloaders(**kwargs)

creating dataloader for:  train
creating dataloader for:  val
creating dataloader for:  test


In [6]:
peepholes._loaders


{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad08a750>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4ab59bd640>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aa50ffbc0>}

In [7]:
peepholes._peepds

{'train': TensorDict(
     fields={
         image: MemoryMappedTensor(shape=torch.Size([40000, 3, 224, 224]), device=cpu, dtype=torch.float32, is_shared=True),
         in_activations: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 25088]), device=cpu, dtype=torch.float32, is_shared=True),
                 classifier.3: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch.float32, is_shared=True),
                 features.28: MemoryMappedTensor(shape=torch.Size([40000, 512, 14, 14]), device=cpu, dtype=torch.float32, is_shared=True)},
             batch_size=torch.Size([40000]),
             device=cpu,
             is_shared=False),
         label: MemoryMappedTensor(shape=torch.Size([40000]), device=cpu, dtype=torch.float32, is_shared=True),
         peepholes: TensorDict(
             fields={
                 classifier.0: MemoryMappedTensor(shape=torch.Size([40000, 4096]), device=cpu, dtype=torch

In [8]:
peepholes._n_samples, batch_dict

({'train': 40000, 'val': 10000, 'test': 10000},
 {'train': 40000, 'val': 10000, 'test': 10000})

# Initialize DkNN

In [9]:
neighbors = 75 
percentage = {'train':1,
               'val':1,
               'test':1}
nb_classes = ds.config['num_classes']
layers = model._target_layers
verbose = True

In [10]:
dknn = DkNN(model, 
            nb_classes, 
            neighbors, 
            layers, 
            peepholes, 
            percentage, 
            seed, 
            verbose,
            NearestNeighbor.BACKEND.FALCONN)

---------- DkNN init

## Constructing the NearestNeighbor tables
Constructing table for classifier.0
done!



In [11]:
dknn.peephole._loaders

{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad08a750>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4ab59bd640>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aa50ffbc0>}

In [12]:
peepholes._loaders

{'train': <torch.utils.data.dataloader.DataLoader at 0x7f4aad08a750>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7f4ab59bd640>,
 'test': <torch.utils.data.dataloader.DataLoader at 0x7f4aa50ffbc0>}

In [13]:
list(model._target_layers.keys()), list(percentage.values())[:-1]

(['classifier.0'], [1, 1])

In [14]:
dknn.calibrate()

---------- DkNN calibrate

## Starting calibration of DkNN
DkNN calibration complete


In [15]:
preds_knn, confs, creds, p_value, test_labels = dknn.fprop('test')

---------- DkNN predict

Nonconformity calculated
Predictions created


In [16]:
torch.Tensor(confs)

tensor([0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0707, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707,
        0.0101, 0.0101, 0.0707, 0.0101, 0.0101, 0.0101, 0.0707, 0.0707, 0.0101,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707,
        0.0101, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707, 0.0101, 0.0101, 0.0707,
        0.0101, 0.0707, 0.0707, 0.0707, 0.0707, 0.0101, 0.0707, 0.0707, 0.0707,
        0.0707, 0.0101, 0.0707, 0.0101, 0.0101, 0.0707, 0.0101, 0.0707, 0.0707,
        0.0707])

In [17]:
peepholes._file_paths

{'train': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.train'),
 'val': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.val'),
 'test': PosixPath('/home/lorenzocapelli/repos/XAI/src/../data/peepholes/peepholes.test')}

In [18]:
preds_knn, confs, creds, p_value, test_labels = dknn.fprop('train')

---------- DkNN predict

Nonconformity calculated
Predictions created


In [19]:
confs.shape, p_value.shape

((400,), (400, 100))

In [20]:
peepholes._n_samples

{'train': 40000, 'val': 10000, 'test': 10000}

In [21]:
dknn.fprop(portion='all')

---------- DkNN predict


 ---- Getting activations for train

40000
adding scores tensorDict
adding DkNN tensorDict
Nonconformity calculated


RuntimeError: batch dimension mismatch, got self.batch_size=torch.Size([2]) and value.shape=torch.Size([40000]).

In [23]:
peepds.keys()

dict_keys(['train', 'val', 'test'])

In [27]:
peepds['train']

In [36]:
confs

NameError: name 'confs' is not defined